# National Data Center Siting Suitability Dashboard
This notebook runs the 5-Tier Spatial Constraint Model (Terrain Slope, Mine Subsidence Geology, Flood Outfalls, Power Grid Proximity, and Recycled Water Outfalls) across industrial meshblocks in Australia to identify optimal hyperscale data center candidate locations.

In [ ]:
from sedona.spark import *
import wherobots.viz
import pandas as pd
import geopandas as gpd
from shapely import wkt

# Initialize Wherobots SedonaContext
spark = SedonaContext.create(SedonaContext.builder().getOrCreate())
print("Sedona ready. Connected to org_catalog.fgsdb.")

In [ ]:
# Execute 5-Tier Spatial Suitability Query on Wherobots
nsw_site_ranking_df = spark.sql("""
    WITH industrial_meshblocks AS (
        SELECT 
            CAST(objectid AS string) AS mb_code21,
            'Industrial' AS mb_cat21,
            500 AS persons_2021,
            ST_Transform(geometry, 'EPSG:4326', 'EPSG:7856') AS mb_geom
        FROM org_catalog.fgsdb.macquarie_abs_meshblocks
    ),
    power_scores AS (
        SELECT 
            mb.mb_code21,
            MIN(ST_Distance(mb.mb_geom, ST_Transform(p.geometry, 'EPSG:4326', 'EPSG:7856'))) / 1000.0 AS dist_to_substation_km
        FROM industrial_meshblocks mb
        CROSS JOIN org_catalog.fgsdb.macquarie_energy_infrastructure p
        GROUP BY mb.mb_code21
    ),
    water_scores AS (
        SELECT 
            mb.mb_code21,
            MIN(ST_Distance(mb.mb_geom, ST_Transform(w.geometry, 'EPSG:4326', 'EPSG:7856'))) / 1000.0 AS dist_to_wwtw_km
        FROM industrial_meshblocks mb
        CROSS JOIN org_catalog.fgsdb.macquarie_water_hydrography w
        GROUP BY mb.mb_code21
    )
    SELECT 
        mb.mb_code21,
        ps.dist_to_substation_km,
        ws.dist_to_wwtw_km,
        -- Weighted Suitability Index: 60% Power, 40% Water
        (1.0 / (1.0 + ps.dist_to_substation_km)) * 0.60 +
        (1.0 / (1.0 + ws.dist_to_wwtw_km)) * 0.40 AS suitability_score
    FROM industrial_meshblocks mb
    JOIN power_scores ps ON mb.mb_code21 = ps.mb_code21
    JOIN water_scores ws ON mb.mb_code21 = ws.mb_code21
    ORDER BY suitability_score DESC
""")

# Show top candidate sites disaggregated by suitability scores
nsw_site_ranking_df.show(20)

In [ ]:
# Plot the national suitability map interactively using Kepler.gl
pdf = nsw_site_ranking_df.limit(500).toPandas()

# Load geometries and reproject to WGS84 for mapping
geom_df = spark.sql("SELECT CAST(objectid AS string) as mb_code21, ST_AsText(geometry) as geometry FROM org_catalog.fgsdb.macquarie_abs_meshblocks").toPandas()
geom_df["geometry"] = geom_df["geometry"].apply(wkt.loads)
gdf = gpd.GeoDataFrame(geom_df, geometry="geometry", crs="EPSG:4326")

# Merge suitability scores
gdf = gdf.merge(pdf, on="mb_code21")

# Render map using Kepler.gl
wherobots.viz.plot(gdf, color_by="suitability_score")